**Introducción a la Ciencia de Datos 2026**  
Posgrado en Ciencias de la Computación - CICESE  
Dr. Irvin Hussein López Nava

## ICD - 3: Representación

Sesión 3 del curso. Diapositivas: [icd-03-representacion.pdf](https://github.com/husseinlopez/icd2026/blob/main/clases/icd-03-representacion.pdf)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/husseinlopez/icd2026/blob/main/icd-03-representacion.ipynb)

En clase planteamos un problema y lo dejamos abierto: los datos casi nunca llegan en la forma en que los necesitamos. Traen tipos que no corresponden a lo que miden, categorías escritas de tres maneras distintas, información de más en una sola columna, y con frecuencia están repartidos en varias tablas o acomodados a lo ancho cuando los queremos a lo largo. Ese trabajo va *antes* del análisis y consume buena parte del tiempo de un proyecto.

Este notebook recorre ese problema con las dos herramientas de la sesión. **NumPy** aporta el arreglo homogéneo y las operaciones vectorizadas; **pandas** construye sobre él la tabla con etiquetas, que es la representación con la que trabajaremos el resto del curso.

Usamos dos tablas de un estudio ficticio de sensado con dispositivos vestibles. Están sucias a propósito: cada defecto corresponde a una diapositiva de la sesión.

---

### Preparación

Las dos librerías de la sesión. La convención `np` / `pd` es universal: se escribe así en toda la documentación y en todo el código que van a leer.

In [1]:
import numpy as np
import pandas as pd

print("numpy ", np.__version__)
print("pandas", pd.__version__)

numpy  2.1.3
pandas 2.2.3


### Los datos

Dos tablas de un estudio de sensado: 12 participantes con sus datos de registro, y los pasos diarios medidos por el dispositivo durante cuatro días.

Como en la sesión anterior, hay dos formas de cargarlas:

- **En tu máquina.** Clona el repositorio y abre el notebook desde la carpeta clonada; las rutas a `datos/` funcionan tal cual.
- **En Colab.** Comenta las dos primeras líneas y descomenta las dos siguientes.

In [3]:
# participantes = pd.read_csv("datos/sensores-participantes.csv")
# mediciones = pd.read_csv("datos/sensores-mediciones.csv")
participantes = pd.read_csv("https://raw.githubusercontent.com/husseinlopez/icd2026/main/datos/sensores-participantes.csv")
mediciones = pd.read_csv("https://raw.githubusercontent.com/husseinlopez/icd2026/main/datos/sensores-mediciones.csv")

participantes

,id,nombre,sexo,edad,grupo,dispositivo,fecha_registro,fc_reposo
0,P01,Ana Rivera,F,34.0,medio,Empatica E4,22-01-2019 15:00,68.0
1,P02,Luis Ortega,M,41.0,alto,Empatica E4,22-01-2019 15:20,72.0
2,P03,Marta Ruiz,Femenino,29.0,bajo,Actigraph GT3X,05-02-2019 09:10,61.0
3,P04,Diego Salas,m,55.0,medio,Actigraph GT3X,05-02-2019 09:35,77.0
4,P05,Sofía Nava,f,23.0,alto,Empatica E4,11-03-2019 11:00,NaN
5,P06,Iván Castro,Masculino,38.0,bajo,Fitbit Charge,11-03-2019 11:25,70.0
6,P07,Elena Vidal,F,47.0,medio,Fitbit Charge,03-04-2019 16:05,74.0
7,P08,Hugo Bravo,M,31.0,alto,Empatica E4,03-04-2019 16:30,65.0
8,P09,Nadia Franco,Femenino,26.0,bajo,Actigraph GT3X,17-05-2019 08:45,63.0
9,P10,Tomás Peña,M,62.0,medio,Actigraph GT3X,17-05-2019 09:10,80.0


In [4]:
mediciones

,id,dia_1,dia_2,dia_3,dia_4
0,P01,7421,8110,6890,7550
1,P02,10233,9877,11002,10450
2,P03,5120,4890,5340,5010
3,P04,6780,7100,6540,6990
4,P05,12045,11890,12500,11760
5,P06,4320,4110,4560,4400
6,P07,8890,9120,8650,8980
7,P08,11340,10980,11600,11150
8,P09,5670,5430,5890,5720
9,P10,6210,6540,5980,6350


---

## 1. NumPy: por qué existe

NumPy no es una librería más de cálculo. Es la capa sobre la que están construidas pandas, SciPy, scikit-learn y matplotlib. Vale la pena entender qué gana uno al usarla, porque Python ya trae listas.

El arreglo se ve como una lista, pero no lo es.

In [5]:
lista = [7421, 8110, 6890, 7550]
arreglo = np.array(lista)

print(lista)
print(arreglo)
print(type(lista), "->", type(arreglo))

[7421, 8110, 6890, 7550]
[7421 8110 6890 7550]
<class 'list'> -> <class 'numpy.ndarray'>


**Diferencia 1: el arreglo es homogéneo.** Todos sus elementos comparten un mismo tipo, y ese tipo es explícito. Una lista de Python puede mezclar lo que sea.

In [6]:
print("dtype del arreglo:", arreglo.dtype)
print("bytes por elemento:", arreglo.itemsize)

mixta = [7421, "8110", 6890.0, True]   # una lista acepta esto sin quejarse
print("\ntipos en la lista:", [type(x).__name__ for x in mixta])
print("al convertirla a arreglo:", np.array(mixta).dtype)   # todo se vuelve texto

dtype del arreglo: int64
bytes por elemento: 8

tipos en la lista: ['int', 'str', 'float', 'bool']
al convertirla a arreglo: <U32


Esa homogeneidad es lo que permite guardar el arreglo como un bloque contiguo de memoria y operar sobre él sin revisar el tipo de cada elemento.

**Diferencia 2: las operaciones son vectorizadas.** Se escriben sobre el arreglo completo, sin ciclo.

In [7]:
pasos = np.array([7421, 8110, 6890, 7550])

print("pasos + 100 :", pasos + 100)      # se suma a cada elemento
print("pasos / 1000:", pasos / 1000)
print("pasos > 7000:", pasos > 7000)     # devuelve un arreglo de booleanos

pasos + 100 : [7521 8210 6990 7650]
pasos / 1000: [7.421 8.11  6.89  7.55 ]
pasos > 7000: [ True  True False  True]


Con una lista, cada una de esas tres líneas sería un ciclo `for`. Y la diferencia no es solo de escritura: es de tiempo.

In [8]:
grande = np.random.default_rng(0).integers(0, 20000, size=1_000_000)
lista_grande = grande.tolist()

print("con ciclo de Python:")
%timeit sum(x * 2 for x in lista_grande)
print("\ncon NumPy:")
%timeit grande * 2

con ciclo de Python:
65.7 ms ± 2 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)

con NumPy:
905 µs ± 186 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


Entre uno y dos órdenes de magnitud, según la máquina, sobre el mismo cálculo. NumPy delega el ciclo a código compilado en C; Python lo ejecuta interpretado, elemento por elemento. En un dataset de mil filas da igual; en uno de diez millones decide si el análisis es viable.

**Diferencia 3: dimensiones.** El arreglo se extiende de forma natural a matrices, que es como llega casi cualquier dataset: filas de observaciones, columnas de variables.

In [9]:
m = mediciones[["dia_1", "dia_2", "dia_3", "dia_4"]].to_numpy()

print("forma:", m.shape, " (filas, columnas)")
print("primeras dos filas:\n", m[:2])
print("\nprimera columna:", m[:, 0])

forma: (11, 4)  (filas, columnas)
primeras dos filas:
 [[ 7421  8110  6890  7550]
 [10233  9877 11002 10450]]

primera columna: [ 7421 10233  5120  6780 12045  4320  8890 11340  5670  6210  7890]


Las agregaciones aceptan un eje. `axis=1` recorre las columnas y devuelve un valor por fila; `axis=0` recorre las filas y devuelve un valor por columna.

In [10]:
print("media por participante (axis=1):", m.mean(axis=1).round(0))
print("media por día         (axis=0):", m.mean(axis=0).round(0))
print("media global           (sin eje):", m.mean().round(0))

media por participante (axis=1): [ 7493. 10390.  5090.  6852. 12049.  4348.  8910. 11268.  5678.  6270.
  7872.]
media por día         (axis=0): [7811. 7824. 7873. 7845.]
media global           (sin eje): 7838.0


Una advertencia que van a encontrar seguido: en cuanto hay un dato faltante, las agregaciones normales devuelven `nan`. Para eso existen las versiones *NaN-safe* de la tabla que vimos en clase.

In [11]:
con_faltante = np.array([68.0, 72.0, np.nan, 77.0])

print("np.mean   :", np.mean(con_faltante))      # nan contamina el resultado
print("np.nanmean:", np.nanmean(con_faltante))   # ignora el faltante

np.mean   : nan
np.nanmean: 72.33333333333333


---

## 2. pandas: el arreglo con etiquetas

Un arreglo de NumPy tiene posiciones. Una tabla real tiene *nombres*: la columna se llama `edad`, la fila corresponde al participante `P03`. Eso es lo que agrega pandas, con dos estructuras.

La `Series` es un arreglo unidimensional con un índice.

In [12]:
s = pd.Series([68, 72, 61, 77], index=["P01", "P02", "P03", "P04"], name="fc_reposo")
s

,fc_reposo
P01,68
P02,72
P03,61
P04,77


In [13]:
print("por etiqueta :", s["P03"])
print("por posición  :", s.iloc[2])
print("\nvalores (es un arreglo de NumPy):", type(s.values).__name__, s.values)

por etiqueta : 61
por posición  : 61

valores (es un arreglo de NumPy): ndarray [68 72 61 77]


El `DataFrame` es un conjunto de Series que comparten el mismo índice. Cada columna conserva su propio tipo; por eso una tabla puede ser heterogénea aunque cada columna sea homogénea.

In [14]:
participantes.head(3)

,id,nombre,sexo,edad,grupo,dispositivo,fecha_registro,fc_reposo
0,P01,Ana Rivera,F,34.0,medio,Empatica E4,22-01-2019 15:00,68.0
1,P02,Luis Ortega,M,41.0,alto,Empatica E4,22-01-2019 15:20,72.0
2,P03,Marta Ruiz,Femenino,29.0,bajo,Actigraph GT3X,05-02-2019 09:10,61.0


Conviene poner como índice la columna que identifica a la observación. A partir de ahí, `P03` deja de ser un valor más y pasa a ser una dirección.

In [15]:
participantes = participantes.set_index("id")
participantes.head(3)

,nombre,sexo,edad,grupo,dispositivo,fecha_registro,fc_reposo
id,,,,,,,
P01,Ana Rivera,F,34.0,medio,Empatica E4,22-01-2019 15:00,68.0
P02,Luis Ortega,M,41.0,alto,Empatica E4,22-01-2019 15:20,72.0
P03,Marta Ruiz,Femenino,29.0,bajo,Actigraph GT3X,05-02-2019 09:10,61.0


In [16]:
participantes.loc["P03"]

,P03
nombre,Marta Ruiz
sexo,Femenino
edad,29.0
grupo,bajo
dispositivo,Actigraph GT3X
fecha_registro,05-02-2019 09:10
fc_reposo,61.0


`.loc` selecciona por etiqueta, `.iloc` por posición. La confusión entre ambos es la fuente de errores más común al empezar.

In [17]:
print(participantes.loc["P03", "edad"])   # etiqueta de fila, etiqueta de columna
print(participantes.iloc[2, 2])           # tercera fila, tercera columna

29.0
29.0


---

## 3. El significado de los datos

Antes de tocar nada: ¿qué es cada columna? La tabla sola no lo dice. `nombre` es evidente, pero `grupo` no, y `fc_reposo` menos. Esa información vive en el **diccionario de datos**, y es responsabilidad de quien publica el conjunto.

In [18]:
diccionario = pd.DataFrame([
    ("id",             "texto",  "Identificador del participante. Clave primaria."),
    ("nombre",         "texto",  "Nombre y apellido en un solo campo."),
    ("sexo",           "texto",  "Sexo reportado por el participante."),
    ("edad",           "entero", "Edad en años cumplidos al registro."),
    ("grupo",          "texto",  "Nivel de actividad asignado: bajo < medio < alto."),
    ("dispositivo",    "texto",  "Modelo del sensor vestible utilizado."),
    ("fecha_registro", "texto",  "Fecha y hora de alta, formato dd-mm-aaaa hh:mm."),
    ("fc_reposo",      "entero", "Frecuencia cardiaca en reposo, en latidos por minuto."),
], columns=["columna", "tipo esperado", "descripción"])

diccionario

,columna,tipo esperado,descripción
0,id,texto,Identificador del participante. Clave primaria.
1,nombre,texto,Nombre y apellido en un solo campo.
2,sexo,texto,Sexo reportado por el participante.
3,edad,entero,Edad en años cumplidos al registro.
4,grupo,texto,Nivel de actividad asignado: bajo < medio < alto.
5,dispositivo,texto,Modelo del sensor vestible utilizado.
6,fecha_registro,texto,"Fecha y hora de alta, formato dd-mm-aaaa hh:mm."
7,fc_reposo,entero,"Frecuencia cardiaca en reposo, en latidos por ..."


Compárenlo con lo que pandas *dedujo* al leer el archivo. Las dos columnas no coinciden, y ahí empieza el trabajo de la sesión.

In [19]:
comparacion = diccionario.set_index("columna")[["tipo esperado"]].copy()
comparacion["dtype leido"] = participantes.dtypes.astype(str)
comparacion.loc["id", "dtype leido"] = "(es el índice)"

comparacion

,tipo esperado,dtype leido
columna,,
id,texto,(es el índice)
nombre,texto,object
sexo,texto,object
edad,entero,float64
grupo,texto,object
dispositivo,texto,object
fecha_registro,texto,object
fc_reposo,entero,float64


Tres tipos de desacuerdo, cada uno con su causa:

- `edad` y `fc_reposo` se esperaban enteros y llegaron `float64`. El culpable es el dato faltante de P11 y P05: el tipo `int64` de NumPy no puede representar un `NaN`, así que pandas promueve toda la columna a flotante.
- `fecha_registro` llegó como `object`, es decir, texto. Para pandas es una cadena cualquiera; no puede ordenarla ni restarla.
- `grupo` llegó como `object`, pero sus valores tienen un orden (`bajo < medio < alto`) que el tipo no registra.

Ninguno es un error del archivo. Son la distancia entre lo que el dato *significa* y lo que la computadora *entendió*.

---

## 4. Manejando tipos

### Enteros con faltantes

pandas ofrece un entero *nullable*, `Int64` con mayúscula, que sí admite valores ausentes. Se recupera la semántica de entero sin perder el faltante.

In [20]:
participantes[["edad", "fc_reposo"]] = participantes[["edad", "fc_reposo"]].astype("Int64")

print(participantes[["edad", "fc_reposo"]].dtypes, "\n")
participantes[["edad", "fc_reposo"]].head(6)

edad         Int64
fc_reposo    Int64
dtype: object 



,edad,fc_reposo
id,,
P01,34,68
P02,41,72
P03,29,61
P04,55,77
P05,23,<NA>
P06,38,70


### Fechas

`pd.to_datetime` convierte texto a un tipo de fecha real. Pero aquí hay una trampa que cuesta cara.

In [21]:
ambiguas = pd.Series(["05-02-2019", "03-04-2019"])

pd.to_datetime(ambiguas)   # sin decirle nada

,0
0,2019-05-02
1,2019-03-04


`05-02-2019` es 5 de febrero, y pandas lo leyó como 2 de mayo. Ningún error, ninguna advertencia útil: simplemente asumió el formato estadounidense mes-día-año. Cuando todos los días del mes son menores o iguales a 12, la equivocación es silenciosa y contamina todo el análisis posterior.

La regla es sencilla: **siempre especificar el formato.**

In [22]:
participantes["fecha_registro"] = pd.to_datetime(
    participantes["fecha_registro"], format="%d-%m-%Y %H:%M"
)

print(participantes["fecha_registro"].dtype)
participantes["fecha_registro"].head(3)

datetime64[ns]


,fecha_registro
id,
P01,2019-01-22 15:00:00
P02,2019-01-22 15:20:00
P03,2019-02-05 09:10:00


Con el tipo correcto se abren operaciones que sobre texto no existían.

In [23]:
print("registro más antiguo:", participantes["fecha_registro"].min())
print("más reciente         :", participantes["fecha_registro"].max())
print("rango del estudio    :", participantes["fecha_registro"].max() - participantes["fecha_registro"].min())

participantes["mes"] = participantes["fecha_registro"].dt.month
participantes[["fecha_registro", "mes"]].head(3)

registro más antiguo: 2019-01-22 15:00:00
más reciente         : 2019-06-28 13:40:00
rango del estudio    : 156 days 22:40:00


,fecha_registro,mes
id,,
P01,2019-01-22 15:00:00,1
P02,2019-01-22 15:20:00,1
P03,2019-02-05 09:10:00,2


---

## 5. Tipos categóricos

La diapositiva de la sesión enumera cuatro problemas de las variables categóricas. Los cuatro están en esta tabla.

### Los niveles no están consolidados

In [24]:
participantes["sexo"].value_counts()

,count
sexo,
M,3
F,2
Femenino,2
f,2
Masculino,2
m,1


Seis niveles para dos categorías. Nadie se equivocó al capturar: se capturó en momentos distintos, con criterios distintos. Es lo más común del mundo.

In [25]:
equivalencias = {
    "F": "Femenino", "f": "Femenino", "Femenino": "Femenino",
    "M": "Masculino", "m": "Masculino", "Masculino": "Masculino",
}
participantes["sexo"] = participantes["sexo"].map(equivalencias)

participantes["sexo"].value_counts()

,count
sexo,
Femenino,6
Masculino,6


Antes de mapear conviene siempre mirar `value_counts()`. Si un valor no aparece en el diccionario, `map` lo convierte en `NaN` sin avisar; el conteo posterior lo delata.

### Una sola columna incorpora información distinta

`nombre` guarda dos cosas. Mientras estén juntas no se puede agrupar por apellido, ni ordenar, ni cruzar con otra tabla.

In [26]:
participantes[["nombre_pila", "apellido"]] = (
    participantes["nombre"].str.split(" ", n=1, expand=True)
)

participantes[["nombre", "nombre_pila", "apellido"]].head(4)

,nombre,nombre_pila,apellido
id,,,
P01,Ana Rivera,Ana,Rivera
P02,Luis Ortega,Luis,Ortega
P03,Marta Ruiz,Marta,Ruiz
P04,Diego Salas,Diego,Salas


El accesor `.str` aplica métodos de cadena a toda la columna, elemento por elemento, sin ciclo. Es el equivalente para texto de lo que hace NumPy con los números.

### Los niveles tienen un orden que el tipo no registra

In [27]:
participantes["grupo"].sort_values().unique()   # orden alfabético: no es el que queremos

array(['alto', 'bajo', 'medio'], dtype=object)

`alto, bajo, medio` es el orden del diccionario, no el de la variable. El tipo `category` permite declarar el orden real.

In [28]:
participantes["grupo"] = pd.Categorical(
    participantes["grupo"], categories=["bajo", "medio", "alto"], ordered=True
)

print(participantes["grupo"].dtype)
print(participantes["grupo"].sort_values().unique())

category
['bajo', 'medio', 'alto']
Categories (3, object): ['bajo' < 'medio' < 'alto']


In [29]:
participantes["grupo"] > "bajo"   # ahora la comparación tiene sentido

,grupo
id,
P01,True
P02,True
P03,False
P04,True
P05,True
P06,False
P07,True
P08,True
P09,False


### Convertir categorías a números

Casi todos los algoritmos que veremos en la unidad 4 requieren números. Hay dos maneras de llegar ahí, y la elección importa.

In [30]:
participantes["grupo"].cat.codes   # ordinal: 0 < 1 < 2, respeta el orden declarado

,0
id,
P01,1
P02,2
P03,0
P04,1
P05,2
P06,0
P07,1
P08,2
P09,0


In [31]:
pd.get_dummies(participantes["dispositivo"], prefix="disp")

,disp_Actigraph GT3X,disp_Empatica E4,disp_Fitbit Charge
id,,,
P01,False,True,False
P02,False,True,False
P03,True,False,False
P04,True,False,False
P05,False,True,False
P06,False,False,True
P07,False,False,True
P08,False,True,False
P09,True,False,False


Para `grupo`, que es ordinal, el código numérico es apropiado: la distancia entre `bajo` y `alto` significa algo.

Para `dispositivo`, que es nominal, no lo es. Asignar 0, 1, 2 a tres modelos de sensor le diría al algoritmo que el segundo está *entre* el primero y el tercero, lo cual es falso. Ahí corresponde `get_dummies`, que crea una columna binaria por nivel.

Es la distinción nominal/ordinal de la sesión 2, ahora con consecuencias operativas.

### El número de niveles puede ser abrumador

El cuarto problema no aparece en esta tabla porque es pequeña, pero es el que más estorba en datos reales: una variable con cientos de niveles, casi todos con una o dos ocurrencias. La estrategia habitual es conservar los más frecuentes y agrupar el resto.

In [32]:
frecuentes = participantes["dispositivo"].value_counts()
print(frecuentes, "\n")

top = frecuentes.head(2).index
participantes["dispositivo_agr"] = participantes["dispositivo"].where(
    participantes["dispositivo"].isin(top), other="Otro"
)

participantes["dispositivo_agr"].value_counts()

dispositivo
Empatica E4       5
Actigraph GT3X    4
Fitbit Charge     3
Name: count, dtype: int64 



,count
dispositivo_agr,
Empatica E4,5
Actigraph GT3X,4
Otro,3


---

## 6. Ancho y largo

En clase vimos la caricatura del autobús: los mismos datos, dos vistas distintas. Ese es exactamente el problema de la tabla de mediciones.

In [33]:
mediciones.head(3)

,id,dia_1,dia_2,dia_3,dia_4
0,P01,7421,8110,6890,7550
1,P02,10233,9877,11002,10450
2,P03,5120,4890,5340,5010


Está en formato **ancho**: cada día es una columna. Se lee bien, pero tiene un defecto de fondo. `dia_1` no es una variable; es un *valor* de la variable *día*, encaramado en el encabezado.

En formato **largo** cada fila es una observación y cada columna una variable. Es la forma que esperan las funciones de agregación, de graficación y casi todos los modelos. Se conoce como *tidy data* (Wickham, 2014).

In [34]:
largo = mediciones.melt(
    id_vars="id",
    var_name="dia",
    value_name="pasos",
)

print(largo.shape)
largo.head(6)

(44, 3)


,id,dia,pasos
0,P01,dia_1,7421
1,P02,dia_1,10233
2,P03,dia_1,5120
3,P04,dia_1,6780
4,P05,dia_1,12045
5,P06,dia_1,4320


De 11 filas y 5 columnas a 44 filas y 3. Ahora `dia` es una columna sobre la que se puede filtrar, agrupar y graficar.

In [35]:
largo["dia"] = largo["dia"].str.replace("dia_", "").astype(int)
largo.head(3)

,id,dia,pasos
0,P01,1,7421
1,P02,1,10233
2,P03,1,5120


La operación inversa es `pivot`. Se usa sobre todo al final, para presentar resultados en una tabla legible.

In [36]:
largo.pivot(index="id", columns="dia", values="pasos").head(3)

dia,1,2,3,4
id,,,,
P01,7421,8110,6890,7550
P02,10233,9877,11002,10450
P03,5120,4890,5340,5010


---

## 7. Unir tablas

Las dos tablas hablan de los mismos participantes pero viven separadas. `merge` las cruza por la columna que comparten.

In [37]:
print("ids en participantes:", sorted(participantes.index))
print("ids en mediciones   :", sorted(mediciones['id']))

ids en participantes: ['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12']
ids en mediciones   : ['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P13']


No son los mismos: P11 y P12 no tienen mediciones, y P13 tiene mediciones pero no ficha de registro. Lo que ocurra con esos tres depende del argumento `how`.

In [38]:
p = participantes.reset_index()

for how in ["inner", "left", "right", "outer"]:
    r = p.merge(mediciones, on="id", how=how)
    print(f"{how:6s} -> {len(r):2d} filas")

inner  -> 10 filas
left   -> 12 filas
right  -> 11 filas
outer  -> 13 filas


- `inner` conserva solo lo que está en ambas: se pierden P11, P12 y P13.
- `left` conserva todos los participantes; P13 desaparece.
- `right` conserva todas las mediciones; P11 y P12 desaparecen.
- `outer` no pierde nada y rellena con `NaN` lo que falta.

`inner` es el valor por omisión y es también la forma más silenciosa de perder datos. Vale la pena comparar el número de filas antes y después de cada cruce.

In [39]:
completo = p.merge(largo, on="id", how="inner")

print(completo.shape)
completo[["id", "apellido", "sexo", "edad", "grupo", "dia", "pasos"]].head(6)

(40, 14)


,id,apellido,sexo,edad,grupo,dia,pasos
0,P01,Rivera,Femenino,34,medio,1,7421
1,P01,Rivera,Femenino,34,medio,2,8110
2,P01,Rivera,Femenino,34,medio,3,6890
3,P01,Rivera,Femenino,34,medio,4,7550
4,P02,Ortega,Masculino,41,alto,1,10233
5,P02,Ortega,Masculino,41,alto,2,9877


---

## 8. Agrupar

Con la tabla ya limpia, unida y en formato largo, las preguntas se responden en una línea. Ese es el punto de todo el trabajo anterior.

In [40]:
completo.groupby("grupo", observed=True)["pasos"].mean().round(0)

,pasos
grupo,
bajo,5038.0
medio,7381.0
alto,11236.0


El patrón es siempre el mismo: **dividir** por una clave, **aplicar** una función, **combinar** el resultado.

In [41]:
completo.groupby("grupo", observed=True)["pasos"].agg(["count", "mean", "std", "min", "max"]).round(1)

,count,mean,std,min,max
grupo,,,,,
bajo,12,5038.3,593.7,4110,5890
medio,16,7381.3,1054.4,5980,9120
alto,12,11235.6,780.2,9877,12500


In [42]:
completo.groupby(["grupo", "sexo"], observed=True)["pasos"].mean().round(0).unstack()

sexo,Femenino,Masculino
grupo,,
bajo,5384.0,4348.0
medio,8201.0,6561.0
alto,12049.0,10829.0


Los grupos declarados como `bajo`, `medio` y `alto` se ordenan solos, porque en la sección 5 declaramos ese orden. Si `grupo` siguiera siendo texto, la tabla saldría en orden alfabético.

Y una última mirada al resultado de la sesión: la tabla con la que empezamos, ya utilizable.

In [43]:
completo.dtypes

,0
id,object
nombre,object
sexo,object
edad,Int64
grupo,category
dispositivo,object
fecha_registro,datetime64[ns]
fc_reposo,Int64
mes,int32
nombre_pila,object


---

### Declaración de uso de IA generativa

Este notebook fue elaborado con apoyo de Claude (Anthropic) para estructurar el contenido a partir de las diapositivas de la sesión, redactar el texto explicativo y construir los conjuntos de datos de ejemplo. El diseño pedagógico, la selección de temas y la revisión final son responsabilidad del titular del curso.

### Referencias

- VanderPlas, J. (2022). *Python Data Science Handbook*, 2a ed. O'Reilly. [Capítulo 2: NumPy](https://jakevdp.github.io/PythonDataScienceHandbook/02.00-introduction-to-numpy.html) · [Capítulo 3: pandas](https://jakevdp.github.io/PythonDataScienceHandbook/03.00-introduction-to-pandas.html)
- Harris, C. R. *et al.* (2020). Array programming with NumPy. *Nature*, 585, 357-362.
- McKinney, W. (2010). Data structures for statistical computing in Python. *Proceedings of the 9th Python in Science Conference*.
- Wickham, H. (2014). Tidy data. *Journal of Statistical Software*, 59(10).
- Gebru, T. *et al.* (2021). Datasheets for datasets. *Communications of the ACM*, 64(12), 86-92.
- [NumPy: the absolute basics for beginners](https://numpy.org/doc/stable/user/absolute_beginners.html)
- [pandas: 10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html)